# Study 925 — Front-End Trend — the teardown

**Signal.** `s_t = 1[ y_t − y_{t−63} < 0 ]` where `y` is `^IRX`, the CBOE 13-week
Treasury bill discount yield in percent — a **price-only yield level**, never a return.
`s` is shifted forward one day: exactly **one** execution lag, formed through the close of
`t`, acted at `t+1`, applied in `rate_trend_signal` and nowhere else.

**Book.** `s = 1` → long TLT; `s = 0` → long BIL. Binary, unlevered, **no short leg
anywhere**, so no borrow is paid or assumed. Costs are `cost_bps × 1e-4 × |Δs|` — one-way,
on NAV, on switch days only.

**Race.** All arms excess-of-cash (minus BIL's own total return): the switch, static IEF
(7-10y, the honest static-duration benchmark), static TLT (the instrument held always), a
frequency-matched random control, and a **matched constant-weight blend** carrying the
rule's average exposure with zero flips. Inference: Newey-West HAC *t* on the daily return
*difference* (Jobson-Korkie form), a paired circular block bootstrap on the Sharpe
difference, an era cut, a lookback sweep, a cost sweep, a HAC day-level conditional test,
and a seed-swept random control run gross **and** net.

*Real-tape numbers below are the frozen headline from [`docs/results.md`](../docs/results.md) — SHY/IEF/TLT/BIL total-return closes plus the `^IRX` yield, 2007-05-30 → 2026-06-30 (4,800 days), Fingerprint `c05691fe8719`, as-of 2026-06-30. Live cells run the offline synthetic control only, and are labelled as such.*


In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4800, 'fp': 'c05691fe8719', 'lookback': 63, 'cost_bps': 2.0, 'in_frac': 0.478, 'n_switches': 321, 'switches_per_year': 17, 'sw_sharpe': 0.154, 'sw_sharpe_gross': 0.185, 'sw_cagr': 1.09, 'sw_vol': 10.98, 'sw_dd': -28.2, 'sw_t': 0.73, 'ief_sharpe': 0.293, 'ief_cagr': 1.82, 'ief_vol': 6.99, 'ief_dd': -23.9, 'ief_t': 1.37, 'tlt_sharpe': 0.184, 'tlt_cagr': 1.66, 'tlt_vol': 15.24, 'tlt_dd': -48.4, 'tlt_t': 0.89, 'rnd_sharpe': -0.316, 'rnd_cagr': -3.74, 'rnd_dd': -47.2, 'adv': -0.139, 't_vs_ief': -0.2, 't_vs_tlt': -0.52, 'adv_one_seed': 0.469, 't_one_seed': 2.01, 'ci_sw_lo': -0.283, 'ci_sw_hi': 0.554, 'ci_sw_neg': 24.3, 'ci_ief_lo': -0.136, 'ci_ief_hi': 0.704, 'ci_ief_neg': 9.3, 'ci_diff_lo': -0.498, 'ci_diff_hi': 0.221, 'ci_diff_neg': 76.4, 'rnd_net_adv': 0.268, 'rnd_net_sd': 0.177, 'rnd_net_t': 1.2, 'rnd_net_share': 20, 'rnd_gross_adv': 0.06, 'rnd_gross_sd': 0.175, 'rnd_gross_t': 0.29, 'rnd_gross_share': 0, 'rnd_flips': 2400, 'blend_w': 47.8, 'blend_turnover': 91, 'blend_net_sharpe': 0.182, 'blend_net_adv': -0.028, 'blend_net_t': 0.23, 'blend_gross_sharpe': 0.184, 'blend_gross_adv': 0.001, 'blend_gross_t': 0.43, 'on_bp': 1.69, 'on_n': 2263, 'on_t': 0.87, 'off_bp': 0.59, 'off_n': 2473, 'off_t': 0.36, 'spread_bp': 1.09, 'spread_hac': 0.43, 'spread_welch': 0.39, 'era_e_n': 2100, 'era_e_ief': 0.73, 'era_e_sw': 0.28, 'era_e_adv': -0.452, 'era_e_t': -0.71, 'era_e_frac': 58, 'era_e_dd_ief': -10.4, 'era_e_dd_sw': -27.1, 'era_l_n': 2635, 'era_l_ief': -0.11, 'era_l_sw': 0.03, 'era_l_adv': 0.138, 'era_l_t': 0.44, 'era_l_frac': 39, 'era_l_dd_ief': -23.9, 'era_l_dd_sw': -28.2, 'lb21_adv': 0.056, 'lb21_t': 0.99, 'lb42_adv': -0.178, 'lb42_t': -0.41, 'lb63_adv': -0.139, 'lb63_t': -0.2, 'lb126_adv': -0.122, 'lb126_t': -0.11, 'lb252_adv': 0.083, 'lb252_t': 1.25, 'cost0_adv': -0.108, 'cost0_t': -0.01, 'cost10_adv': -0.264, 'cost10_t': -0.96, 'cost25_adv': -0.496, 'cost25_t': -2.35, 'years': ((2007, 6.7, 5.5, 7.3, 1.2, 99), (2008, 26.1, 17.9, 34.0, 8.2, 78), (2009, -21.5, -6.6, -21.8, -14.9, 67), (2010, -2.4, 9.4, 9.0, -11.7, 47), (2011, 26.4, 15.6, 34.0, 10.7, 73), (2012, 4.5, 3.7, 2.4, 0.8, 20), (2013, -14.8, -6.1, -13.4, -8.7, 59), (2014, 24.9, 9.1, 27.3, 15.9, 73), (2015, -11.3, 1.5, -1.8, -12.8, 38), (2016, 2.7, 1.0, 1.2, 1.7, 24), (2017, -1.4, 2.6, 9.2, -3.9, 2), (2018, 1.7, 1.0, -1.6, 0.7, 0), (2019, 6.9, 8.0, 14.1, -1.2, 67), (2020, 6.1, 10.0, 18.2, -3.9, 83), (2021, -15.2, -3.3, -4.6, -11.9, 56), (2022, 1.4, -15.2, -31.2, 16.5, 0), (2023, 21.0, 3.6, 2.8, 17.3, 15), (2024, -9.4, -0.6, -8.1, -8.8, 71), (2025, 8.8, 8.0, 4.2, 0.8, 72), (2026, 1.8, -0.1, 1.0, 1.9, 47)), 'yrs_beat': 11, 'yrs_total': 20, 'gap_mean': -0.1, 'gap_median': 0.76, 'y2022_sw': 1.4, 'y2022_ief': -15.2, 'y2022_tlt': -31.2, 'y2022_sessions': 0, 'y2022_total': 251, 'y2023_sw': 21.0, 'y2023_ief': 3.6, 'y2023_days': 37, 'y2023_tlt_window': 14.7, 'y2023_strat_window': 16.2, 'y2023_bil': 4.9, 'shy_sharpe': 0.331, 'shy_adv': 0.061, 'shy_t': 1.43, 'shy_frac': 85, 'shy_switches': 68, 'shy_gross_rnd': 0.172, 'shy_gross_rnd_t': 1.36, 'shy_blend_w': 85.2, 'shy_blend_sharpe': 0.177, 'shy_blend_adv': 0.153, 'shy_blend_t': 1.92, 'shy_blend_gross_adv': 0.158, 'shy_blend_gross_t': 1.97, 'shy_blend_early_adv': 0.005, 'shy_blend_early_t': 0.51, 'shy_blend_early_frac': 99, 'shy_blend_late_adv': 0.243, 'shy_blend_late_t': 1.52, 'shy_blend_ex22_adv': 0.156, 'shy_blend_ex22_t': 1.24, 'syn_blend_adv': 1.545, 'syn_blend_t': 4.45, 'syn_pl_adv': 1.78, 'syn_pl_t': 4.66, 'syn_pl_mean': 2.15, 'syn_pl_fire': 10, 'syn_nl_mean': -0.05, 'syn_nl_sd': 0.203, 'syn_nl_fire': 0}
print(f"frozen headline: {R['start']} -> {R['end']}, n={R['n_days']}, fp={R['fp']}")
print(f"in duration {R['in_frac']:.1%} of days, {R['n_switches']} switches "
      f"(~{R['switches_per_year']}/yr)")

frozen headline: 2007-05-30 -> 2026-06-30, n=4800, fp=c05691fe8719
in duration 47.8% of days, 321 switches (~17/yr)


## 1. The excess-of-cash race

> 💡 **In plain words** — every arm has T-bill returns subtracted, so nobody gets paid for merely sitting in cash while cash yielded 5%.

In [2]:
print(f"{'arm':16s}{'exSharpe':>10s}{'exCAGR':>9s}{'vol':>8s}{'maxDD':>9s}{'HAC t':>8s}")
print(f"{'switch':16s}{R['sw_sharpe']:+10.3f}{R['sw_cagr']:+8.2f}%{R['sw_vol']:7.2f}%"
      f"{R['sw_dd']:+8.1f}%{R['sw_t']:+8.2f}")
print(f"{'static IEF':16s}{R['ief_sharpe']:+10.3f}{R['ief_cagr']:+8.2f}%{R['ief_vol']:7.2f}%"
      f"{R['ief_dd']:+8.1f}%{R['ief_t']:+8.2f}")
print(f"{'static TLT':16s}{R['tlt_sharpe']:+10.3f}{R['tlt_cagr']:+8.2f}%{R['tlt_vol']:7.2f}%"
      f"{R['tlt_dd']:+8.1f}%{R['tlt_t']:+8.2f}")
print(f"{'random (1 seed)':16s}{R['rnd_sharpe']:+10.3f}{R['rnd_cagr']:+8.2f}%"
      f"{'':8s}{R['rnd_dd']:+8.1f}%")
print()
print(f"adv vs static IEF : {R['adv']:+.3f}   HAC t on daily return diff = {R['t_vs_ief']:+.2f}")
print(f"adv vs static TLT :         HAC t on daily return diff = {R['t_vs_tlt']:+.2f}")
print('the switch arm is worse on Sharpe, MORE volatile and DEEPER in drawdown than IEF.')

arm               exSharpe   exCAGR     vol    maxDD   HAC t
switch              +0.154   +1.09%  10.98%   -28.2%   +0.73
static IEF          +0.293   +1.82%   6.99%   -23.9%   +1.37
static TLT          +0.184   +1.66%  15.24%   -48.4%   +0.89
random (1 seed)     -0.316   -3.74%           -47.2%

adv vs static IEF : -0.139   HAC t on daily return diff = -0.20
adv vs static TLT :         HAC t on daily return diff = -0.52
the switch arm is worse on Sharpe, MORE volatile and DEEPER in drawdown than IEF.


## 2. Paired block bootstrap on the Sharpe *difference*

2,000 draws, 21-day circular blocks, **same block indices drawn for both arms** so the paired structure and the shared cash leg survive resampling.

In [3]:
print(f"switch exSharpe      {R['sw_sharpe']:+.3f}  95% CI [{R['ci_sw_lo']:+.3f}, {R['ci_sw_hi']:+.3f}]  share<0 {R['ci_sw_neg']:.1f}%")
print(f"static IEF exSharpe  {R['ief_sharpe']:+.3f}  95% CI [{R['ci_ief_lo']:+.3f}, {R['ci_ief_hi']:+.3f}]  share<0 {R['ci_ief_neg']:.1f}%")
print(f"DIFFERENCE           {R['adv']:+.3f}  95% CI [{R['ci_diff_lo']:+.3f}, {R['ci_diff_hi']:+.3f}]  share<0 {R['ci_diff_neg']:.1f}%")
print('the difference CI straddles zero with 76% of the mass on the losing side.')

switch exSharpe      +0.154  95% CI [-0.283, +0.554]  share<0 24.3%
static IEF exSharpe  +0.293  95% CI [-0.136, +0.704]  share<0 9.3%
DIFFERENCE           -0.139  95% CI [-0.498, +0.221]  share<0 76.4%
the difference CI straddles zero with 76% of the mass on the losing side.


## 3. The random control is a turnover trap

A frequency-matched Bernoulli control flips roughly `2p(1−p)N` times. With `p = 0.478` and `N = 4,800` that is ~2,400 flips against the rule's 321 — so at any positive cost the control pays ~7× the friction it never chose. One seed is also one draw. Sweep 30 seeds, and run it **gross**.

> 💡 **In plain words** — the rule did not pick better days than the coin. It just traded less often than the coin, and we were charging the coin for that.

In [4]:
print(f"single seed (the tempting number): adv {R['adv_one_seed']:+.3f}  t {R['t_one_seed']:+.2f}")
print()
print(f"{'':10s}{'adv mean':>10s}{'sd':>8s}{'t mean':>9s}{'share t>=2':>12s}")
print(f"{'net 2bps':10s}{R['rnd_net_adv']:+10.3f}{R['rnd_net_sd']:8.3f}"
      f"{R['rnd_net_t']:+9.2f}{R['rnd_net_share']:>11d}%")
print(f"{'GROSS':10s}{R['rnd_gross_adv']:+10.3f}{R['rnd_gross_sd']:8.3f}"
      f"{R['rnd_gross_t']:+9.2f}{R['rnd_gross_share']:>11d}%")
print()
print(f"~{R['rnd_flips']:,} control flips vs {R['n_switches']} rule switches -> "
      'the net advantage is friction accounting, not timing.')

single seed (the tempting number): adv +0.469  t +2.01

            adv mean      sd   t mean  share t>=2
net 2bps      +0.268   0.177    +1.20         20%
GROSS         +0.060   0.175    +0.29          0%

~2,400 control flips vs 321 rule switches -> the net advantage is friction accounting, not timing.


### 3b. The control the random one should have been

Remove the friction confound entirely: hold the rule's **average** exposure with **no flips at all** — a constant `w = 47.8%` TLT / 52.2% BIL book rebalanced daily (drift trades only, ~91%/yr of turnover, charged at the same cost the rule pays). Same duration on average, no regime switching, no seed to draw. `strategy.matched_blend_race` does this in one call.

> 💡 **In plain words** — if the timing is worth anything, it has to beat simply owning that average all the time.

In [5]:
print(f"{'':10s}{'switch':>10s}{'blend':>10s}{'adv':>10s}{'ret-diff t':>12s}")
print(f"{'net 2bps':10s}{R['sw_sharpe']:+10.3f}{R['blend_net_sharpe']:+10.3f}"
      f"{R['blend_net_adv']:+10.3f}{R['blend_net_t']:+12.2f}")
print(f"{'GROSS':10s}{R['sw_sharpe_gross']:+10.3f}"
      f"{R['blend_gross_sharpe']:+10.3f}{R['blend_gross_adv']:+10.3f}"
      f"{R['blend_gross_t']:+12.2f}")
print()
print('gross: the same number to two decimals. net: the flip-free blend wins.')
print('321 timing decisions in nineteen years are collectively worth zero.')

              switch     blend       adv  ret-diff t
net 2bps      +0.154    +0.182    -0.028       +0.23
GROSS         +0.185    +0.184    +0.001       +0.43

gross: the same number to two decimals. net: the flip-free blend wins.
321 timing decisions in nineteen years are collectively worth zero.


## 4. Day-level conditional test — does the signal carry *any* information?

Strip out portfolio construction and cost entirely: take the excess-of-cash daily return of TLT and split it by what the rule said the day before.

The spread's *t* is the **HAC** one — the Newey-West *t* of `e·(s − p)/(p(1−p))`, whose mean is exactly the difference in bucket means. Welch's iid *t* is printed for reference only: the buckets are selected by a signal that persists for months, so daily returns inside a bucket are not independent draws.

In [6]:
print(f"own-duration days : {R['on_bp']:+.2f} bp/d  n={R['on_n']}  HAC t {R['on_t']:+.2f}")
print(f"sit-in-bills days : {R['off_bp']:+.2f} bp/d  n={R['off_n']}  HAC t {R['off_t']:+.2f}")
print(f"spread            : {R['spread_bp']:+.2f} bp/d   HAC t = {R['spread_hac']:+.2f}"
      f"   (Welch t = {R['spread_welch']:+.2f}, iid, reference only)")
print('right sign, no significance -- 4,736 daily observations cannot separate the buckets.')

own-duration days : +1.69 bp/d  n=2263  HAC t +0.87
sit-in-bills days : +0.59 bp/d  n=2473  HAC t +0.36
spread            : +1.09 bp/d   HAC t = +0.43   (Welch t = +0.39, iid, reference only)
right sign, no significance -- 4,736 daily observations cannot separate the buckets.


## 5. Era cut and lookback sweep — the sign is not stable

The signal is built on the full yield history then sliced, so the late era does not pay a fresh 63-day warmup (that would be a different rule, not a robustness check); no look-ahead is introduced because the signal still uses past yields only.

In [7]:
print(f"2007-2015 (n={R['era_e_n']}): IEF {R['era_e_ief']:+.2f} / switch {R['era_e_sw']:+.2f}  "
      f"adv {R['era_e_adv']:+.3f} (t={R['era_e_t']:+.2f})  in-duration {R['era_e_frac']}%")
print(f"2016-2026 (n={R['era_l_n']}): IEF {R['era_l_ief']:+.2f} / switch {R['era_l_sw']:+.2f}  "
      f"adv {R['era_l_adv']:+.3f} (t={R['era_l_t']:+.2f})  in-duration {R['era_l_frac']}%")
print('  -> sign flips between halves; neither half is significant.')
print()
for lb, a, t in [(21, R['lb21_adv'], R['lb21_t']), (42, R['lb42_adv'], R['lb42_t']),
                 (63, R['lb63_adv'], R['lb63_t']), (126, R['lb126_adv'], R['lb126_t']),
                 (252, R['lb252_adv'], R['lb252_t'])]:
    mark = '  <- headline' if lb == 63 else ''
    print(f'lookback {lb:3d}d: adv {a:+.3f} (t={t:+.2f}){mark}')
print('  -> three sign changes across a reasonable grid. The conclusion is the knob.')

2007-2015 (n=2100): IEF +0.73 / switch +0.28  adv -0.452 (t=-0.71)  in-duration 58%
2016-2026 (n=2635): IEF -0.11 / switch +0.03  adv +0.138 (t=+0.44)  in-duration 39%
  -> sign flips between halves; neither half is significant.

lookback  21d: adv +0.056 (t=+0.99)
lookback  42d: adv -0.178 (t=-0.41)
lookback  63d: adv -0.139 (t=-0.20)  <- headline
lookback 126d: adv -0.122 (t=-0.11)
lookback 252d: adv +0.083 (t=+1.25)
  -> three sign changes across a reasonable grid. The conclusion is the knob.


## 6. Cost sweep and the SHY cross-check

No short leg, so no borrow: the only friction is switch cost, one-way × NAV. The cross-check swaps the `^IRX` yield change for **SHY's 12-minus-1-month total-return momentum** — the same bet read off a tradable instrument instead of an index yield.

In [8]:
print(f"gross (0 bps): adv {R['cost0_adv']:+.3f} (t={R['cost0_t']:+.2f})  <- already negative")
print(f"10 bps       : adv {R['cost10_adv']:+.3f} (t={R['cost10_t']:+.2f})")
print(f"25 bps       : adv {R['cost25_adv']:+.3f} (t={R['cost25_t']:+.2f})  <- the only |t|>=2 here, wrong way")
print()
print(f"SHY 12-1 momentum variant: switch {R['shy_sharpe']:+.3f}, adv {R['shy_adv']:+.3f} "
      f"(t={R['shy_t']:+.2f}), in-duration {R['shy_frac']}%, {R['shy_switches']} switches")
print(f"  gross vs random over 30 seeds: {R['shy_gross_rnd']:+.3f} (t={R['shy_gross_rnd_t']:+.2f})")
print(f"  vs ITS matched blend (w={R['shy_blend_w']:.1f}% TLT): "
      f"adv {R['shy_blend_adv']:+.3f} (t={R['shy_blend_t']:+.2f}) net, "
      f"{R['shy_blend_gross_adv']:+.3f} (t={R['shy_blend_gross_t']:+.2f}) GROSS")
print(f"    2007-2015 (in duration {R['shy_blend_early_frac']}% of days): "
      f"adv {R['shy_blend_early_adv']:+.3f} (t={R['shy_blend_early_t']:+.2f})")
print(f"    2016-2026: adv {R['shy_blend_late_adv']:+.3f} (t={R['shy_blend_late_t']:+.2f})"
      f"   |  full sample minus 2022: adv {R['shy_blend_ex22_adv']:+.3f} "
      f"(t={R['shy_blend_ex22_t']:+.2f})")

gross (0 bps): adv -0.108 (t=-0.01)  <- already negative
10 bps       : adv -0.264 (t=-0.96)
25 bps       : adv -0.496 (t=-2.35)  <- the only |t|>=2 here, wrong way

SHY 12-1 momentum variant: switch +0.331, adv +0.061 (t=+1.43), in-duration 85%, 68 switches
  gross vs random over 30 seeds: +0.172 (t=+1.36)
  vs ITS matched blend (w=85.2% TLT): adv +0.153 (t=+1.92) net, +0.158 (t=+1.97) GROSS
    2007-2015 (in duration 99% of days): adv +0.005 (t=+0.51)
    2016-2026: adv +0.243 (t=+1.52)   |  full sample minus 2022: adv +0.156 (t=+1.24)


**The strongest number in the study, and why it is still not a badge.** The SHY variant beats a fixed-weight book of its *own* average exposure by **+0.153** net and **+0.158** gross (*t* = +1.92 / +1.97) — not a turnover artefact, and the right comparison for a rule that is long duration 85% of the time. It is still under |*t*| ≥ 2, and it is not stable: in 2007-2015 the rule held duration on 99% of days — it made no decisions to be right about — and its advantage there is +0.005. Everything it has comes from the second half, where the 2022 hiking cycle dominates; drop 2022 and the *t* falls to +1.24. One alternative specification, one era, under the bar. Interesting; not evidence.

## 7. Live synthetic control — the harness is unbiased (offline)

The knob is the AR(1) coefficient `phi` on the short rate's daily *increments*, with the unconditional daily rate volatility rescaled by `sqrt(1 − phi²)` so the planted and null worlds carry identical risk and differ only in predictability. Duration legs are priced off the same path with `r = carry/252 − D·dy + noise`.

> 💡 **In plain words** — we prove the detector works by planting the effect and watching it fire, then removing it and watching it go quiet.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from front_end_trend import data, strategy as st

pl = st.synthetic_detect(data.synthetic_daily(signal_strength=1.0, seed=925)[0])
print('SYNTHETIC (never supports the real-tape stamp)')
print(f"planted trending front end: adv {pl['excess_sharpe_adv']:+.3f} (t={pl['t_diff_vs_mid']:+.2f})")
nl = np.array([st.synthetic_detect(f)['excess_sharpe_adv']
               for f, _ in data.synthetic_panel(n_seeds=10, signal_strength=0.0)])
pp = np.array([st.synthetic_detect(f)['excess_sharpe_adv']
               for f, _ in data.synthetic_panel(n_seeds=10, signal_strength=1.0)])
print(f"planted x10: mean {pp.mean():+.3f}, adv>=0.4 in {(pp >= 0.4).sum()}/10")
print(f"null    x10: mean {nl.mean():+.3f} (sd {nl.std(ddof=1):.3f}), "
      f"|adv|>=0.4 in {(np.abs(nl) >= 0.4).sum()}/10")

cd_pl = st.conditional_day_test(*[data.synthetic_daily(signal_strength=1.0, seed=925)[0][c]
                                  for c in ('rate', 'long', 'cash')])
cd_nl = st.conditional_day_test(*[data.synthetic_daily(signal_strength=0.0, seed=925)[0][c]
                                  for c in ('rate', 'long', 'cash')])
print(f"day-level spread  planted {cd_pl['spread_bp']:+.2f} bp/d "
      f"(HAC t {cd_pl['spread_hac_t']:+.2f})   null {cd_nl['spread_bp']:+.2f} bp/d "
      f"(HAC t {cd_nl['spread_hac_t']:+.2f})")

sig_pl = st.rate_trend_signal(data.synthetic_daily(signal_strength=1.0, seed=925)[0]['rate'])
f_pl = data.synthetic_daily(signal_strength=1.0, seed=925)[0]
mb = st.matched_blend_race(f_pl['long'], f_pl['mid'], f_pl['cash'], sig_pl)
print(f"matched-blend control on the planted world: adv {mb['adv_vs_blend']:+.3f} "
      f"(t {mb['t_diff_vs_blend']:+.2f}) -> real timing does beat its own average exposure")

SYNTHETIC (never supports the real-tape stamp)
planted trending front end: adv +1.784 (t=+4.66)


planted x10: mean +2.145, adv>=0.4 in 10/10
null    x10: mean -0.050 (sd 0.203), |adv|>=0.4 in 0/10


day-level spread  planted +19.78 bp/d (HAC t +4.47)   null +1.59 bp/d (HAC t +0.87)


matched-blend control on the planted world: adv +1.545 (t +4.45) -> real timing does beat its own average exposure


## Verdict

- **Signal — None.** Excess-of-cash Sharpe advantage over static IEF is **-0.139** (HAC *t* = -0.20); versus static TLT the difference *t* is -0.52. The paired bootstrap difference CI is **[-0.498, +0.221]** with 76% of draws negative. The sign flips across eras (-0.45 / +0.14) and three times across the lookback grid. The day-level conditional spread is +1.09 bp/d at HAC *t* = +0.43. The apparent win over a random control is a **turnover artefact**: +0.268 net but only **+0.060 gross** across 30 seeds (mean *t* +0.29, 0/30 clearing *t* ≥ 2); against the flip-free blend of identical exposure it is **+0.001 gross, -0.028 net**. Nothing clears |*t*| ≥ 2 in the claim's direction. The synthetic control fires at adv **+2.15** (10/10) on a planted trending front end and sits at **-0.050** (0/10) on the null, so the blank is the tape's, not the harness's; the blend control separates there too (adv +1.545, *t* +4.45).
- **The one number that argues back.** The SHY 12-1 variant beats *its* matched blend by +0.153 net / +0.158 gross (*t* +1.92 / +1.97) — the best result on this desk's tape for the idea, under the bar, and worth +0.005 in the era where the rule held duration 99% of days.
- **Tradability — Mirage.** Lower excess Sharpe, higher vol (11.0% vs 7.0%), deeper drawdown (-28.2% vs -23.9%), ~17 round trips a year, negative gross and more negative at every cost tested, ahead of IEF in 11/20 calendar years for a mean gap of -0.10 pp/yr. Nothing to bank.
- **The honest residue.** 2022 (0 of 251 sessions in duration, +16.5 points over IEF) and late 2023 (37 sessions long while TLT rose +14.7%) are real, and they are the whole *timing* record: the other big winning years (2011 +10.7, 2014 +15.9) are 20-year duration beta the rule rode up and then paid back in 2009, 2015 and 2021. Two episodes in nineteen years is not a sample; it is the shape luck takes when it is loud.